In [1]:
from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt

from contact_plots import get_data_from_files, create_quartile_plot

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder, convert_2_minutes=True)

In [3]:
cut_off_distances = ['1.0', '1.5', '2.0']

data_names_base = (
    'customer_customer_weight_over_contacts',
    'customer_customer_weight_over_agents',
    'customer_customer_contacts_over_agents',
    'staff_customer_weight_over_contacts',
    'staff_customer_weight_over_agents',
    'staff_customer_contacts_over_agents',
    'customer_staff_weight_over_agents',
    'customer_staff_contacts_over_agents',
)

parts_2_data_name = {}
for cut_off_distance in cut_off_distances:
    data_name_dict = {}
    for data_name in data_names_base:
        parts = data_name.split('_')
        interaction_type = '_'.join(parts[:2])
        data_type = '_'.join(parts[2:])
        if interaction_type not in data_name_dict:
            data_name_dict[interaction_type] = {}
            
        data_name_dict[interaction_type][data_type] = f'{cut_off_distance}_{data_name}'
    
    parts_2_data_name[cut_off_distance] = data_name_dict
    
measures_and_labels = (
    ('nn_nts_bvd', 'Base'),
    ('n_nts_bvd', 'Neighbourhoods'),
    ('nn_ts_bvd', 'Time slots'),
    ('nn_nts_svd', 'Shorter visits'),
    ('nn_nts_lvd', 'Longer visits')
)

base_measure_name = 'nn_nts_bvd'

infra_names = [
    'base_case',
    'busy_corner',
    'long_restaurant']

In [4]:
divider_config = {
    'left':0.95,  
    'gap_hor':1.25,
    'right':0.1,
    'bottom':1.15,
    'gap_ver':0.30,
    'v_legend':0.50,
    'top':0.10,    
}

measure_names = [measure_name for measure_name, _ in measures_and_labels]
labels = [label for _, label in measures_and_labels]


#bar_plot_config_base = {'width':0.8, 'linewidth':1, 'edgecolor':'k'}
line_config_base = {'linewidth':1, 'marker':'x', 'markersize':8}
legend_config = {'loc':'upper center', 'ncol':6}

ax_config = {'grid':{'visible':True, 'axis':'y'}, 'y_label_config':{'fontsize':10},
                 'x_tick_labels':labels, 'x_tick_labels_config':{'fontsize':10, 'rotation':45, 'ha':'right'}}

base_plot_config = {'measure_names':measure_names, 'ax_config':ax_config, 'legend_config':legend_config, 'gap_size':0.4}

plot_types_to_y_labels_and_titles = (
    ('weight_over_contacts', 'Weight [min]', 'Weight over contacts'),
    ('weight_over_agents', 'Weight [min]', 'Weight over agents'),
    ('contacts_over_agents', 'Count [-]', 'Contacts over agent')
)

In [5]:
%matplotlib wx

plt.style.use('ggplot')

infra_2_style = {
    'base_case':{'color':'tab:blue'},
    'busy_corner':{'color':'tab:red'},
    'long_restaurant':{'color':'tab:green'}    
}

interaction_type_all = 'customer_customer'
cut_off_dist = '1.5'

data_name_slice = parts_2_data_name[cut_off_dist][interaction_type_all]

fig_config = {'title': f'Quartile - {interaction_type_all}'}

plot_config = deepcopy(base_plot_config)
plots_config = []

for data_type, y_label, title in plot_types_to_y_labels_and_titles:
    plot_config_sub = deepcopy(plot_config)
    line_plot_configs = []
    for infra_name in infra_names:
        line_config = deepcopy(line_config_base)
        line_config.update(infra_2_style[infra_name])
        line_plot_config = {'infra_name':infra_name, 
                          'data_name':data_name_slice[data_type], 
                          'label':infra_name,
                          'line_config':line_config}
        
        line_plot_configs.append(line_plot_config)
    
    plot_config_sub['line_plot_configs'] = line_plot_configs
    plot_config_sub['ax_config'].update({'title':title, 'y_label':y_label})
    plots_config.append(plot_config_sub)
    
create_quartile_plot(fig_config, divider_config, plots_config, infra_and_measure_name_2_data)

In [6]:
plt.show()